In [ ]:
# ============================================================
# MONTHLY + SEASONAL DELTA_T vs MTBF ANALYSIS
# ============================================================
# DATASET:
# PCB_MTBF_Processed.xlsx
#
# PARAMETER:
# Delta_T
#
# ANALYSIS INCLUDED:
#
# 1. Region-wise Monthly Analysis
# 2. SLoc-wise Monthly Analysis
# 3. Region-wise Seasonal Analysis
# 4. SLoc-wise Seasonal Analysis
# 5. Pearson Correlation
# 6. Spearman Correlation
# 7. Kendall Tau Correlation
# 8. ANOVA
# 9. Kruskal Wallis
# 10. Tukey HSD
# 11. Chi Square Test
# 12. Cramer's V
# ============================================================

# ============================================================
# IMPORT LIBRARIES
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd

from scipy.stats import (
    pearsonr,
    spearmanr,
    kendalltau,
    f_oneway,
    kruskal,
    chi2_contingency
)

from statsmodels.stats.multicomp import pairwise_tukeyhsd

warnings.filterwarnings("ignore")

# ============================================================
# BASE DIRECTORY
# ============================================================

base_directory = r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\100\MTBF\Delta_T_Analysis"

# ============================================================
# CREATE FOLDERS
# ============================================================

analysis_types = [
    "Monthly_Analysis",
    "Seasonal_Analysis"
]

subfolders = [
    "Region_Wise",
    "SLoc_Wise",
    "Correlation_Results",
    "ANOVA",
    "Kruskal_Wallis",
    "Tukey_HSD",
    "Chi_Square",
    "Cramers_V"
]

for analysis in analysis_types:

    for folder in subfolders:

        os.makedirs(
            os.path.join(base_directory, analysis, folder),
            exist_ok=True
        )

print("Folders Created Successfully")


file_path = r"C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\100\MTBF\PCB_MTBF_Processed.xlsx"

df = pd.read_excel(file_path)

print("\nDataset Loaded Successfully")
print("Dataset Shape :", df.shape)

df.columns = df.columns.str.strip()

df['Region'] = df['Region'].astype(str)
df['SLoc'] = df['SLoc'].astype(str)
df['Season'] = df['Season'].astype(str)

if 'DeltaT_Band' not in df.columns:

    df['DeltaT_Band'] = pd.qcut(
        df['Delta_T'],
        q=3,
        labels=['Low', 'Medium', 'High']
    )

df = df.dropna(
    subset=[
        'MTBF_Days',
        'Delta_T',
        'Region',
        'SLoc'
    ]
)

print("\nAfter Removing Missing Values:")
print(df.shape)

# ============================================================
# FUNCTION : CORRELATION ANALYSIS
# ============================================================

def correlation_analysis(
    dataframe,
    group_column,
    parameter_column,
    output_path
):

    results = []

    grouped = dataframe.groupby(group_column)

    for group_name, group_data in grouped:

        if len(group_data) < 3:
            continue

        x = group_data[parameter_column]
        y = group_data['MTBF_Days']

        # ====================================================
        # PEARSON
        # ====================================================

        try:
            pearson_corr, pearson_p = pearsonr(x, y)

        except:
            pearson_corr, pearson_p = np.nan, np.nan

        # ====================================================
        # SPEARMAN
        # ====================================================

        try:
            spearman_corr, spearman_p = spearmanr(x, y)

        except:
            spearman_corr, spearman_p = np.nan, np.nan

        # ====================================================
        # KENDALL
        # ====================================================

        try:
            kendall_corr, kendall_p = kendalltau(x, y)

        except:
            kendall_corr, kendall_p = np.nan, np.nan

        # ====================================================
        # STORE RESULTS
        # ====================================================

        results.append({

            group_column: group_name,

            'Pearson_Correlation': pearson_corr,
            'Pearson_P_Value': pearson_p,

            'Spearman_Correlation': spearman_corr,
            'Spearman_P_Value': spearman_p,

            'KendallTau_Correlation': kendall_corr,
            'KendallTau_P_Value': kendall_p
        })

    results_df = pd.DataFrame(results)

    results_df.to_excel(output_path, index=False)

    print(f"\nSaved : {output_path}")

# ============================================================
# FUNCTION : ANOVA
# ============================================================

def perform_anova(
    dataframe,
    group_column,
    output_path
):

    grouped_data = []

    for name, group in dataframe.groupby(group_column):

        values = group['MTBF_Days'].dropna()

        if len(values) > 1:
            grouped_data.append(values)

    if len(grouped_data) < 2:

        print(f"Not enough groups for ANOVA : {group_column}")
        return

    f_stat, p_value = f_oneway(*grouped_data)

    results_df = pd.DataFrame({

        'F_Statistic': [f_stat],
        'P_Value': [p_value]

    })

    results_df.to_excel(output_path, index=False)

    print(f"\nSaved : {output_path}")

# ============================================================
# FUNCTION : KRUSKAL WALLIS
# ============================================================

def perform_kruskal(
    dataframe,
    group_column,
    output_path
):

    grouped_data = []

    for name, group in dataframe.groupby(group_column):

        values = group['MTBF_Days'].dropna()

        if len(values) > 1:
            grouped_data.append(values)

    if len(grouped_data) < 2:

        print(f"Not enough groups for Kruskal : {group_column}")
        return

    h_stat, p_value = kruskal(*grouped_data)

    results_df = pd.DataFrame({

        'H_Statistic': [h_stat],
        'P_Value': [p_value]

    })

    results_df.to_excel(output_path, index=False)

    print(f"\nSaved : {output_path}")

# ============================================================
# FUNCTION : TUKEY HSD
# ============================================================

def perform_tukey(
    dataframe,
    group_column,
    output_path
):

    dataframe[group_column] = (
        dataframe[group_column]
        .astype(str)
    )

    dataframe = dataframe.dropna(
        subset=['MTBF_Days', group_column]
    )

    tukey = pairwise_tukeyhsd(

        endog=dataframe['MTBF_Days'],
        groups=dataframe[group_column],
        alpha=0.05

    )

    tukey_df = pd.DataFrame(

        data=tukey.summary().data[1:],
        columns=tukey.summary().data[0]

    )

    tukey_df.to_excel(output_path, index=False)

    print(f"\nSaved : {output_path}")

# ============================================================
# FUNCTION : CHI-SQUARE + CRAMER'S V
# ============================================================

def chi_square_analysis(
    dataframe,
    output_path
):

    contingency_table = pd.crosstab(

        dataframe['DeltaT_Band'],
        dataframe['MTBF_Category']

    )

    # ========================================================
    # CHI-SQUARE
    # ========================================================

    chi2, p, dof, expected = chi2_contingency(
        contingency_table
    )

    # ========================================================
    # CRAMER'S V
    # ========================================================

    n = contingency_table.sum().sum()

    cramers_v = np.sqrt(

        chi2 / (
            n * (
                min(contingency_table.shape) - 1
            )
        )
    )

    results_df = pd.DataFrame({

        'Chi2_Statistic': [chi2],
        'P_Value': [p],
        'Degrees_of_Freedom': [dof],
        'Cramers_V': [cramers_v]

    })

    results_df.to_excel(output_path, index=False)

    print(f"\nSaved : {output_path}")

# ============================================================
# MONTHLY ANALYSIS
# ============================================================

monthly_base = os.path.join(
    base_directory,
    "Monthly_Analysis"
)

# ============================================================
# REGION-WISE MONTHLY
# ============================================================

region_monthly = (
    df.groupby(
        ['Region', 'Year', 'Month']
    )
    .agg({
        'MTBF_Days': 'mean',
        'Delta_T': 'mean'
    })
    .reset_index()
)

region_monthly.to_excel(
    os.path.join(
        monthly_base,
        "Region_Wise",
        "Region_Wise_Monthly_Data.xlsx"
    ),
    index=False
)

# ============================================================
# SLOC-WISE MONTHLY
# ============================================================

sloc_monthly = (
    df.groupby(
        ['SLoc', 'Year', 'Month']
    )
    .agg({
        'MTBF_Days': 'mean',
        'Delta_T': 'mean'
    })
    .reset_index()
)

sloc_monthly.to_excel(
    os.path.join(
        monthly_base,
        "SLoc_Wise",
        "SLoc_Wise_Monthly_Data.xlsx"
    ),
    index=False
)

# ============================================================
# MONTHLY CORRELATIONS
# ============================================================

correlation_analysis(
    region_monthly,
    'Region',
    'Delta_T',
    os.path.join(
        monthly_base,
        "Correlation_Results",
        "Region_Wise_Correlation.xlsx"
    )
)

correlation_analysis(
    sloc_monthly,
    'SLoc',
    'Delta_T',
    os.path.join(
        monthly_base,
        "Correlation_Results",
        "SLoc_Wise_Correlation.xlsx"
    )
)

# ============================================================
# MONTHLY ANOVA
# ============================================================

perform_anova(
    region_monthly,
    'Region',
    os.path.join(
        monthly_base,
        "ANOVA",
        "Region_Wise_ANOVA.xlsx"
    )
)

perform_anova(
    sloc_monthly,
    'SLoc',
    os.path.join(
        monthly_base,
        "ANOVA",
        "SLoc_Wise_ANOVA.xlsx"
    )
)

# ============================================================
# MONTHLY KRUSKAL
# ============================================================

perform_kruskal(
    region_monthly,
    'Region',
    os.path.join(
        monthly_base,
        "Kruskal_Wallis",
        "Region_Wise_Kruskal.xlsx"
    )
)

perform_kruskal(
    sloc_monthly,
    'SLoc',
    os.path.join(
        monthly_base,
        "Kruskal_Wallis",
        "SLoc_Wise_Kruskal.xlsx"
    )
)

# ============================================================
# MONTHLY TUKEY
# ============================================================

perform_tukey(
    region_monthly,
    'Region',
    os.path.join(
        monthly_base,
        "Tukey_HSD",
        "Region_Wise_Tukey.xlsx"
    )
)

perform_tukey(
    sloc_monthly,
    'SLoc',
    os.path.join(
        monthly_base,
        "Tukey_HSD",
        "SLoc_Wise_Tukey.xlsx"
    )
)

# ============================================================
# MONTHLY CHI-SQUARE
# ============================================================

chi_square_analysis(
    df,
    os.path.join(
        monthly_base,
        "Chi_Square",
        "DeltaT_vs_MTBF_ChiSquare.xlsx"
    )
)

# ============================================================
# SEASONAL ANALYSIS
# ============================================================

seasonal_base = os.path.join(
    base_directory,
    "Seasonal_Analysis"
)

# ============================================================
# REGION-WISE SEASONAL
# ============================================================

region_seasonal = (
    df.groupby(
        ['Region', 'Season', 'Year']
    )
    .agg({
        'MTBF_Days': 'mean',
        'Delta_T': 'mean'
    })
    .reset_index()
)

region_seasonal.to_excel(
    os.path.join(
        seasonal_base,
        "Region_Wise",
        "Region_Wise_Seasonal_Data.xlsx"
    ),
    index=False
)

# ============================================================
# SLOC-WISE SEASONAL
# ============================================================

sloc_seasonal = (
    df.groupby(
        ['SLoc', 'Season', 'Year']
    )
    .agg({
        'MTBF_Days': 'mean',
        'Delta_T': 'mean'
    })
    .reset_index()
)

sloc_seasonal.to_excel(
    os.path.join(
        seasonal_base,
        "SLoc_Wise",
        "SLoc_Wise_Seasonal_Data.xlsx"
    ),
    index=False
)

# ============================================================
# SEASONAL CORRELATIONS
# ============================================================

correlation_analysis(
    region_seasonal,
    'Region',
    'Delta_T',
    os.path.join(
        seasonal_base,
        "Correlation_Results",
        "Region_Wise_Seasonal_Correlation.xlsx"
    )
)

correlation_analysis(
    sloc_seasonal,
    'SLoc',
    'Delta_T',
    os.path.join(
        seasonal_base,
        "Correlation_Results",
        "SLoc_Wise_Seasonal_Correlation.xlsx"
    )
)

# ============================================================
# SEASONAL ANOVA
# ============================================================

perform_anova(
    region_seasonal,
    'Region',
    os.path.join(
        seasonal_base,
        "ANOVA",
        "Region_Wise_Seasonal_ANOVA.xlsx"
    )
)

perform_anova(
    sloc_seasonal,
    'SLoc',
    os.path.join(
        seasonal_base,
        "ANOVA",
        "SLoc_Wise_Seasonal_ANOVA.xlsx"
    )
)

# ============================================================
# SEASONAL KRUSKAL
# ============================================================

perform_kruskal(
    region_seasonal,
    'Region',
    os.path.join(
        seasonal_base,
        "Kruskal_Wallis",
        "Region_Wise_Seasonal_Kruskal.xlsx"
    )
)

perform_kruskal(
    sloc_seasonal,
    'SLoc',
    os.path.join(
        seasonal_base,
        "Kruskal_Wallis",
        "SLoc_Wise_Seasonal_Kruskal.xlsx"
    )
)

# ============================================================
# SEASONAL TUKEY
# ============================================================

perform_tukey(
    region_seasonal,
    'Region',
    os.path.join(
        seasonal_base,
        "Tukey_HSD",
        "Region_Wise_Seasonal_Tukey.xlsx"
    )
)

perform_tukey(
    sloc_seasonal,
    'SLoc',
    os.path.join(
        seasonal_base,
        "Tukey_HSD",
        "SLoc_Wise_Seasonal_Tukey.xlsx"
    )
)

# ============================================================
# SEASONAL CHI-SQUARE
# ============================================================

chi_square_analysis(
    df,
    os.path.join(
        seasonal_base,
        "Chi_Square",
        "DeltaT_vs_MTBF_Seasonal_ChiSquare.xlsx"
    )
)

# ============================================================
# COMPLETED
# ============================================================

print("\n===================================================")
print("DELTA_T ANALYSIS COMPLETED SUCCESSFULLY")
print("===================================================")

Folders Created Successfully

Dataset Loaded Successfully
Dataset Shape : (9805, 21)

After Removing Missing Values:
(9805, 22)

Saved : C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\100\MTBF\Delta_T_Analysis\Monthly_Analysis\Correlation_Results\Region_Wise_Correlation.xlsx

Saved : C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\100\MTBF\Delta_T_Analysis\Monthly_Analysis\Correlation_Results\SLoc_Wise_Correlation.xlsx

Saved : C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\100\MTBF\Delta_T_Analysis\Monthly_Analysis\ANOVA\Region_Wise_ANOVA.xlsx

Saved : C:\Users\Amey\OneDrive - College of Engineering Pune(An autonomous Institute of Govt. of Maharashtra)\Desktop\Amey\Python\100\MTBF\Delta_T_Analysis\Monthly_Analysis\ANOVA\SLoc_Wise_ANOVA.xlsx

Saved : C:\Users\Amey\O